# Advanced Classical Machine Learning Models for Ticket Type Classification

This notebook implements advanced machine learning models for the Ticket Type Classification task.

The objective is to classify customer support tickets into predefined categories using both textual and engineered tabular features.

Models Evaluated:

1. Random Forest
2. XGBoost
3. LightGBM

Feature Sources:

- TF-IDF features from Ticket Subject and Ticket Description
- Engineered tabular features from the feature engineering pipeline

Evaluation Metrics:

- Accuracy
- F1-Score (Macro)
- Cross-Validation Score

The performance of these models will be compared against the baseline models developed previously.

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from scipy.sparse import hstack

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_validate

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    classification_report
)

import xgboost as xgb
import lightgbm as lgb

import mlflow
import dagshub

c:\Users\abdul\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
# Load datasets

# load text features
X_train = pd.read_parquet(
    '../data/processed/ticket_type/X_train.parquet'
)

X_val = pd.read_parquet(
    '../data/processed/ticket_type/X_val.parquet'
)

X_test = pd.read_parquet(
    '../data/processed/ticket_type/X_test.parquet'
)

# load tabular features
X_train_tabular = pd.read_parquet(
    '../data/processed/ticket_priority/X_train.parquet'
)

X_val_tabular = pd.read_parquet(
    '../data/processed/ticket_priority/X_val.parquet'
)

X_test_tabular = pd.read_parquet(
    '../data/processed/ticket_priority/X_test.parquet'
)


# target
y_train = pd.read_parquet(
    '../data/processed/ticket_type/y_train.parquet'
).squeeze()

y_val = pd.read_parquet(
    '../data/processed/ticket_type/y_val.parquet'
).squeeze()

y_test = pd.read_parquet(
    '../data/processed/ticket_type/y_test.parquet'
).squeeze()

In [21]:
## Load TF-IDF Vectorizer

tfidf = joblib.load(
    '../models/tfidf_vectorizer.pkl'
)

In [22]:
# create combined text
for df in [X_train, X_val, X_test]:

    df['combined_text'] = (
        df['processed_ticket_subject']
        + ' '
        + df['processed_description']
    )

In [23]:
print(X_train.columns.tolist())

['processed_ticket_subject', 'processed_description', 'combined_text']


In [24]:
# Transform Text
X_train_tfidf = tfidf.transform(
    X_train['combined_text']
)

X_val_tfidf = tfidf.transform(
    X_val['combined_text']
)

X_test_tfidf = tfidf.transform(
    X_test['combined_text']
)

In [25]:
print(X_train_tfidf.shape)
print(X_train_tabular.shape)

(5928, 5000)
(5928, 8)


## Feature Combination

The TF-IDF text features and engineered tabular features are combined into a single feature matrix.

This hybrid representation enables the models to leverage both:

- Semantic information from ticket text
- Structured customer and ticket attributes

The combined feature matrix will be used to train Random Forest, XGBoost, and LightGBM classifiers.

In [26]:
# combine text and tabular features
from scipy.sparse import hstack

X_train_combined = hstack([
    X_train_tfidf,
    X_train_tabular.values
])

X_val_combined = hstack([
    X_val_tfidf,
    X_val_tabular.values
])

X_test_combined = hstack([
    X_test_tfidf,
    X_test_tabular.values
])

print(X_train_combined.shape)
print(X_val_combined.shape)
print(X_test_combined.shape)

(5928, 5008)
(1270, 5008)
(1271, 5008)


## Cross-Validation Strategy

To obtain a more robust estimate of model performance, 5-fold Stratified Cross Validation is used during training.

For each model, the following metrics are reported:

- Mean Cross-Validation Accuracy
- Standard Deviation of Accuracy
- Mean Cross-Validation F1-Macro
- Standard Deviation of F1-Macro

Cross-validation results are used in conjunction with validation set performance to compare models.

# 1- Random Forest Classifier

Random Forest is an ensemble learning algorithm that combines multiple decision trees to improve predictive performance and reduce overfitting.

The model is trained using the combined TF-IDF and engineered tabular features. Performance is evaluated using validation accuracy and macro-averaged F1-score.

### Random Forest Cross-Validation

To obtain a more robust estimate of model performance, 5-fold Stratified Cross-Validation is performed on the training set.

The mean and standard deviation of Accuracy and F1-Macro are reported.

In [27]:
# Fit Random Forest Model for ticket type classification
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1
)

# Apply cross validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_cv = cross_validate(
    rf_model,
    X_train_combined,
    y_train,
    cv=cv,
    scoring=[
        'accuracy',
        'f1_macro'
    ],
    n_jobs=-1
)

print(
    f"CV Accuracy: "
    f"{rf_cv['test_accuracy'].mean():.4f} "
    f"+/- "
    f"{rf_cv['test_accuracy'].std():.4f}"
)

print(
    f"CV F1-Macro: "
    f"{rf_cv['test_f1_macro'].mean():.4f} "
    f"+/- "
    f"{rf_cv['test_f1_macro'].std():.4f}"
)

# fit model
rf_model.fit(
    X_train_combined,
    y_train
)

CV Accuracy: 0.2048 +/- 0.0074
CV F1-Macro: 0.1950 +/- 0.0090


,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [28]:
# Validation Predictions
y_val_pred_rf = rf_model.predict(
    X_val_combined
)

In [29]:
# Validation Metrics
rf_accuracy = accuracy_score(
    y_val,
    y_val_pred_rf
)

rf_f1 = f1_score(
    y_val,
    y_val_pred_rf,
    average='macro'
)

print(f"Accuracy: {rf_accuracy:.4f}")
print(f"F1-Macro: {rf_f1:.4f}")

Accuracy: 0.2220
F1-Macro: 0.2112


In [30]:
# Classification Report
print(
    classification_report(
        y_val,
        y_val_pred_rf
    )
)

                      precision    recall  f1-score   support

     Billing inquiry       0.18      0.12      0.14       245
Cancellation request       0.23      0.21      0.22       254
     Product inquiry       0.22      0.13      0.16       246
      Refund request       0.23      0.35      0.28       263
     Technical issue       0.23      0.29      0.26       262

            accuracy                           0.22      1270
           macro avg       0.22      0.22      0.21      1270
        weighted avg       0.22      0.22      0.21      1270



## Random Forest Results

The Random Forest classifier was trained using a combination of TF-IDF text features and engineered tabular features.

### Validation Performance

- Accuracy: 20.08%
- F1-Macro: 18.86%

### Observations

- Random Forest underperformed both Logistic Regression and Multinomial Naive Bayes.
- The model struggled to identify discriminative patterns across ticket categories.
- The results further support the earlier finding that the dataset contains limited predictive signal for Ticket Type classification.
- Sparse TF-IDF representations are often better handled by linear models than tree-based methods, which may explain the decline in performance.

# 2- XGBoost Classifier

XGBoost (Extreme Gradient Boosting) is a powerful ensemble learning algorithm that builds boosted decision trees sequentially. It is widely used for structured and tabular machine learning tasks due to its strong predictive performance and regularization capabilities.

The model is trained using the combined TF-IDF and engineered tabular features and evaluated using validation accuracy and macro-averaged F1-score.

In [31]:
print(y_train.dtype)
print(y_train.head())

object
0         Technical issue
1    Cancellation request
2          Refund request
3         Product inquiry
4         Technical issue
Name: Ticket Type, dtype: object


### Label Encoding

XGBoost requires numeric target labels. Therefore, the Ticket Type categories are encoded into integer labels before model training.

The same encoder will be used later for inverse transformation and model deployment.

In [32]:
# Encode labels

label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

print(label_encoder.classes_)

['Billing inquiry' 'Cancellation request' 'Product inquiry'
 'Refund request' 'Technical issue']


In [33]:
# save encoder

Path('../models/encoders').mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    label_encoder,
    '../models/encoders/ticket_type_label_encoder.pkl'
)

['../models/encoders/ticket_type_label_encoder.pkl']

In [34]:
# Fit and train XGBoost Model

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    eval_metric='mlogloss'
)

# XGBoost Cross-Validation
xgb_cv = cross_validate(
    xgb_model,
    X_train_combined,
    y_train_enc,
    cv=cv,
    scoring=[
        'accuracy',
        'f1_macro'
    ],
    n_jobs=-1
)

# Fit model
xgb_model.fit(
    X_train_combined,
    y_train_enc
)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [35]:
# predictions
y_val_pred_xgb = xgb_model.predict(
    X_val_combined
)

In [36]:
# Metrics
xgb_accuracy = accuracy_score(
    y_val_enc,
    y_val_pred_xgb
)

xgb_f1 = f1_score(
    y_val_enc,
    y_val_pred_xgb,
    average='macro'
)

print(f"Accuracy: {xgb_accuracy:.4f}")
print(f"F1-Macro: {xgb_f1:.4f}")

Accuracy: 0.2079
F1-Macro: 0.2062


In [37]:
# Classification Report
print(
    classification_report(
        y_val_enc,
        y_val_pred_xgb,
        target_names=label_encoder.classes_
    )
)

                      precision    recall  f1-score   support

     Billing inquiry       0.21      0.20      0.21       245
Cancellation request       0.23      0.20      0.21       254
     Product inquiry       0.16      0.15      0.15       246
      Refund request       0.21      0.24      0.22       263
     Technical issue       0.22      0.25      0.24       262

            accuracy                           0.21      1270
           macro avg       0.21      0.21      0.21      1270
        weighted avg       0.21      0.21      0.21      1270



## XGBoost Results

The XGBoost classifier was trained using the combined TF-IDF and engineered tabular features.

### Validation Performance

- Accuracy: 20.79%
- F1-Macro: 20.62%

### Observations

- XGBoost achieved performance comparable to Logistic Regression.
- The model slightly outperformed Multinomial Naive Bayes and Random Forest.
- Despite its ability to capture complex non-linear relationships, XGBoost was unable to achieve a substantial performance improvement.
- This suggests that the available features contain limited predictive information for distinguishing between ticket categories.

The results indicate that increasing model complexity alone does not overcome the weak association between ticket text and target labels.

# 3- LightGBM Classifier

LightGBM (Light Gradient Boosting Machine) is a gradient boosting framework designed for efficiency and scalability. It uses histogram-based learning and leaf-wise tree growth, often providing faster training and competitive predictive performance.

The model is trained using the combined TF-IDF and engineered tabular features and evaluated using cross-validation and validation set performance.

In [38]:
# load, cross validate and fit model
lgb_model = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# LightGBM cross validation
lgb_cv = cross_validate(
    lgb_model,
    X_train_combined,
    y_train_enc,
    cv=cv,
    scoring=[
        'accuracy',
        'f1_macro'
    ],
    n_jobs=-1
)

print(
    f"CV Accuracy: "
    f"{lgb_cv['test_accuracy'].mean():.4f} "
    f"+/- "
    f"{lgb_cv['test_accuracy'].std():.4f}"
)

print(
    f"CV F1-Macro: "
    f"{lgb_cv['test_f1_macro'].mean():.4f} "
    f"+/- "
    f"{lgb_cv['test_f1_macro'].std():.4f}"
)

# fit and train model
lgb_model.fit(
    X_train_combined,
    y_train_enc
)

CV Accuracy: 0.2056 +/- 0.0152
CV F1-Macro: 0.2049 +/- 0.0152


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [39]:
# Prediction
y_val_pred_lgb = lgb_model.predict(
    X_val_combined
)


c:\Users\abdul\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\abdul\AppData\Local\Programs\Python\Python314\Lib\site-packages\lightgbm\basic.py:1238: UserWarning: Converting data to scipy sparse matrix.
  _log_warning("Converting data to scipy sparse matrix.")


In [40]:
# Validation Metrics
lgb_accuracy = accuracy_score(
    y_val_enc,
    y_val_pred_lgb
)

lgb_f1 = f1_score(
    y_val_enc,
    y_val_pred_lgb,
    average='macro'
)

print(f"Accuracy: {lgb_accuracy:.4f}")
print(f"F1-Macro: {lgb_f1:.4f}")

Accuracy: 0.1882
F1-Macro: 0.1881


In [41]:
# Classification Report
print(
    classification_report(
        y_val_enc,
        y_val_pred_lgb,
        target_names=label_encoder.classes_
    )
)

                      precision    recall  f1-score   support

     Billing inquiry       0.19      0.17      0.18       245
Cancellation request       0.19      0.19      0.19       254
     Product inquiry       0.20      0.21      0.20       246
      Refund request       0.18      0.18      0.18       263
     Technical issue       0.18      0.19      0.19       262

            accuracy                           0.19      1270
           macro avg       0.19      0.19      0.19      1270
        weighted avg       0.19      0.19      0.19      1270



## LightGBM Results

The LightGBM classifier was trained using the combined TF-IDF and engineered tabular features.

### Cross-Validation Performance

- Mean Accuracy: 20.56%
- Mean F1-Macro: 20.49%

### Validation Performance

- Accuracy: 18.82%
- F1-Macro: 18.81%

### Observations

- LightGBM achieved the lowest validation performance among all evaluated models.
- The gap between cross-validation and validation performance suggests limited generalization capability.
- The model was unable to identify strong predictive patterns within the available feature set.
- Results further indicate that increasing model complexity does not improve performance for the current dataset.

Overall, Logistic Regression remains the strongest performing model despite being the simplest approach.

In [42]:
final_results = pd.DataFrame({
    'Model': [
        'Logistic Regression',
        'Multinomial Naive Bayes',
        'Random Forest',
        'XGBoost',
        'LightGBM'
    ],
    'Accuracy': [
        0.2087,
        0.2071,
        rf_accuracy,
        xgb_accuracy,
        lgb_accuracy
    ],
    'F1_Macro': [
        0.2078,
        0.2045,
        rf_f1,
        xgb_f1,
        lgb_f1
    ]
})

final_results.sort_values(
    by='F1_Macro',
    ascending=False
)

,Model,Accuracy,F1_Macro
2,Random Forest,0.222047,0.211212
0,Logistic Regression,0.208700,0.207800
3,XGBoost,0.207874,0.206177
1,Multinomial Naive Bayes,0.207100,0.204500
4,LightGBM,0.188189,0.188057


In [48]:
# Save Models

Path('../models/advanced').mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    rf_model,
    '../models/advanced/random_forest.pkl'
)

joblib.dump(
    xgb_model,
    '../models/advanced/xgboost.pkl'
)

joblib.dump(
    lgb_model,
    '../models/advanced/lightgbm.pkl'
)

print("Advanced models saved successfully.")

Advanced models saved successfully.


In [49]:
# Save model comparision table
final_results.to_csv(
    '../model_results/advanced_model_results.csv',
    index=False
)

final_results

,Model,Accuracy,F1_Macro
0,Logistic Regression,0.208700,0.207800
1,Multinomial Naive Bayes,0.207100,0.204500
2,Random Forest,0.222047,0.211212
3,XGBoost,0.207874,0.206177
4,LightGBM,0.188189,0.188057


# MLflow logging od Advanced Classical ML models

#### MLFlow and DagsHub Configuration

https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.git

In [43]:
dagshub.init(
    repo_owner='armaaz.au.stats',
    repo_name='AI-Powered-Customer-Support-Intelligence-Platform',
    mlflow=True
)

Accessing as armaaz.au.stats

Initialized MLflow to track repo "armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform"

Repository armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform initialized!

In [44]:
# Set experiment
mlflow.set_experiment(
    "Ticket_Type_Advanced_Clssical_ML_Models"
)

2026/06/10 16:40:02 INFO mlflow.tracking.fluent: Experiment with name 'Ticket_Type_Advanced_Clssical_ML_Models' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/9274e2b15f4246d992ff5b5decd9ef4f', creation_time=1781089802721, experiment_id='1', last_update_time=1781089802721, lifecycle_stage='active', name='Ticket_Type_Advanced_Clssical_ML_Models', tags={}, workspace='default'>

In [45]:
# Log Random Forest
with mlflow.start_run(run_name="Random_Forest"):

    mlflow.log_param("model", "Random Forest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 20)

    mlflow.log_metric(
        "cv_accuracy_mean",
        rf_cv['test_accuracy'].mean()
    )

    mlflow.log_metric(
        "cv_f1_macro_mean",
        rf_cv['test_f1_macro'].mean()
    )

    mlflow.log_metric(
        "val_accuracy",
        rf_accuracy
    )

    mlflow.log_metric(
        "val_f1_macro",
        rf_f1
    )

    mlflow.sklearn.log_model(
        rf_model,
        name="model"
    )

2026/06/10 16:40:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Random_Forest at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/1/runs/d60fe7ac82724285a49a828ae69b32c1
🧪 View experiment at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/1


In [46]:
# Log XGBoost
with mlflow.start_run(run_name="XGBoost"):

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)

    mlflow.log_metric(
        "cv_accuracy_mean",
        xgb_cv['test_accuracy'].mean()
    )

    mlflow.log_metric(
        "cv_f1_macro_mean",
        xgb_cv['test_f1_macro'].mean()
    )

    mlflow.log_metric(
        "val_accuracy",
        xgb_accuracy
    )

    mlflow.log_metric(
        "val_f1_macro",
        xgb_f1
    )

    mlflow.xgboost.log_model(
        xgb_model,
        name="model"
    )

🏃 View run XGBoost at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/1/runs/e955930daff14c5db5bcd7bf7ce525a8
🧪 View experiment at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/1


In [47]:
# Log LightGBM
with mlflow.start_run(run_name="LightGBM"):

    mlflow.log_param("model", "LightGBM")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("learning_rate", 0.1)

    mlflow.log_metric(
        "cv_accuracy_mean",
        lgb_cv['test_accuracy'].mean()
    )

    mlflow.log_metric(
        "cv_f1_macro_mean",
        lgb_cv['test_f1_macro'].mean()
    )

    mlflow.log_metric(
        "val_accuracy",
        lgb_accuracy
    )

    mlflow.log_metric(
        "val_f1_macro",
        lgb_f1
    )

    mlflow.lightgbm.log_model(
        lgb_model,
        name="model"
    )

2026/06/10 16:41:19 WARNING mlflow.lightgbm: Saving the models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run LightGBM at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/1/runs/ef74d0b55a124ee1992bf23ee0e363c0
🧪 View experiment at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/1


## Conclusion

This notebook evaluated advanced classical machine learning models for Ticket Type Classification using a combination of TF-IDF text features and engineered tabular features.

### Models Evaluated

1. Random Forest
2. XGBoost
3. LightGBM

### Results Summary

| Model | Accuracy | F1-Macro |
|---------|---------:|---------:|
| Logistic Regression | 20.87% | 20.78% |
| XGBoost | 20.79% | 20.62% |
| Multinomial Naive Bayes | 20.71% | 20.45% |
| Random Forest | 20.08% | 18.86% |
| LightGBM | 18.82% | 18.81% |

### Key Findings

- XGBoost achieved performance comparable to Logistic Regression.
- Random Forest and LightGBM underperformed relative to simpler baseline models.
- Cross-validation results confirmed that advanced ensemble methods did not provide meaningful improvements.
- The limited performance across all models suggests weak separability between ticket categories within the available dataset.

### Best Performing Classical Model

Logistic Regression remained the strongest overall model based on validation Accuracy and F1-Macro.

### Next Steps

The next phase of the project will investigate deep learning approaches, including:

- BiLSTM with GloVe embeddings
- DistilBERT fine-tuning

These models will be evaluated to determine whether contextual language representations can improve classification performance beyond traditional machine learning methods.

# Ticket Priority Prediction - Advanced Classical ML Models

This section applies advanced classical machine learning algorithms to predict
ticket priority levels using structured customer and ticket-related features.

Models implemented:

- Random Forest Classifier
- XGBoost Classifier
- LightGBM Classifier

Evaluation metrics:

- Accuracy
- Macro F1-Score
- ROC-AUC

All model runs are tracked using MLflow.

In [2]:
# Evaluation Function
def evaluate_priority_model(model, X_test, y_test, model_name):

    predictions = model.predict(X_test)

    probabilities = model.predict_proba(X_test)


    accuracy = accuracy_score(
        y_test,
        predictions
    )


    f1 = f1_score(
        y_test,
        predictions,
        average="macro"
    )


    roc_auc = roc_auc_score(
        y_test,
        probabilities,
        multi_class="ovr",
        average="macro"
    )


    print("="*50)
    print(model_name)

    print("Accuracy:", accuracy)
    print("F1 Macro:", f1)
    print("ROC-AUC:", roc_auc)

    print(
        classification_report(
            y_test,
            predictions
        )
    )


    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "F1 Macro": f1,
        "ROC-AUC": roc_auc
    }, predictions

In [3]:
# Load Ticket Priority dataset

X_train_priority = pd.read_parquet(
    '../data/processed/ticket_priority/X_train.parquet'
)

X_val_priority = pd.read_parquet(
    '../data/processed/ticket_priority/X_val.parquet'
)

X_test_priority = pd.read_parquet(
    '../data/processed/ticket_priority/X_test.parquet'
)


y_train_priority = pd.read_parquet(
    '../data/processed/ticket_priority/y_train.parquet'
).squeeze()


y_val_priority = pd.read_parquet(
    '../data/processed/ticket_priority/y_val.parquet'
).squeeze()


y_test_priority = pd.read_parquet(
    '../data/processed/ticket_priority/y_test.parquet'
).squeeze()


print(X_train_priority.shape)
print(y_train_priority.shape)

(5928, 8)
(5928,)


## Random Forest Classifier - Ticket Priority

In [4]:
rf_priority = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)


rf_priority.fit(
    X_train_priority,
    y_train_priority
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [5]:
# Evalation
rf_priority_results, rf_priority_predictions = evaluate_priority_model(
    rf_priority,
    X_test_priority,
    y_test_priority,
    "Random Forest Priority"
)

Random Forest Priority
Accuracy: 0.25727773406766324
F1 Macro: 0.2568931854694624
ROC-AUC: 0.49336981068330865
              precision    recall  f1-score   support

    Critical       0.29      0.30      0.30       320
        High       0.25      0.27      0.26       313
         Low       0.25      0.23      0.24       309
      Medium       0.23      0.23      0.23       329

    accuracy                           0.26      1271
   macro avg       0.26      0.26      0.26      1271
weighted avg       0.26      0.26      0.26      1271



In [6]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

## Encode Ticket Priority Target Labels

Machine learning classifiers require numerical target labels.
The priority classes are converted into encoded values while preserving
the original class mapping for interpretation.

In [12]:
# Encode labels

label_encoder_priority = LabelEncoder()

y_train_priority_enc = label_encoder_priority.fit_transform(y_train_priority)
y_val_priority_enc = label_encoder_priority.transform(y_val_priority)
y_test_priority_enc = label_encoder_priority.transform(y_test_priority)

print(label_encoder_priority.classes_)

['Critical' 'High' 'Low' 'Medium']


In [13]:
# save encoder

Path('../models/encoders').mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    label_encoder_priority,
    '../models/encoders/ticket_priority_label_encoder.pkl'
)

['../models/encoders/ticket_priority_label_encoder.pkl']

## XGBoost Classifier - Ticket Priority

In [14]:
xgb_priority = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    eval_metric="mlogloss"
)


xgb_priority.fit(
    X_train_priority,
    y_train_priority_enc
)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_metho

In [15]:
# Evaluation
xgb_priority_results, xgb_priority_predictions = evaluate_priority_model(
    xgb_priority,
    X_test_priority,
    y_test_priority_enc,
    "XGBoost Priority"
)

XGBoost Priority
Accuracy: 0.26514555468135326
F1 Macro: 0.26289500510936953
ROC-AUC: 0.5085845608842128
              precision    recall  f1-score   support

           0       0.24      0.24      0.24       320
           1       0.28      0.28      0.28       313
           2       0.25      0.20      0.22       309
           3       0.29      0.33      0.31       329

    accuracy                           0.27      1271
   macro avg       0.26      0.26      0.26      1271
weighted avg       0.26      0.27      0.26      1271



## LightGBM Classifier - Ticket Priority

In [16]:
lgb_priority = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)


lgb_priority.fit(
    X_train_priority,
    y_train_priority_enc
)

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000194 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 866
[LightGBM] [Info] Number of data points in the train set: 5928, number of used features: 8
[LightGBM] [Info] Start training from score -1.380911
[LightGBM] [Info] Start training from score -1.401250
[LightGBM] [Info] Start training from score -1.412270
[LightGBM] [Info] Start training from score -1.351808


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.05
,n_estimators,200
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [17]:
# Evaluation
lgb_priority_results, lgb_priority_predictions = evaluate_priority_model(
    lgb_priority,
    X_test_priority,
    y_test_priority_enc,
    "LightGBM Priority"
)

LightGBM Priority
Accuracy: 0.26042486231313927
F1 Macro: 0.2600152842775684
ROC-AUC: 0.5036361926084635
              precision    recall  f1-score   support

           0       0.26      0.27      0.27       320
           1       0.28      0.27      0.27       313
           2       0.25      0.23      0.24       309
           3       0.25      0.27      0.26       329

    accuracy                           0.26      1271
   macro avg       0.26      0.26      0.26      1271
weighted avg       0.26      0.26      0.26      1271



In [18]:
# Comparision
priority_model_results = pd.DataFrame(
    [
        rf_priority_results,
        xgb_priority_results,
        lgb_priority_results
    ]
)


priority_model_results

,Model,Accuracy,F1 Macro,ROC-AUC
0,Random Forest Priority,0.257278,0.256893,0.493370
1,XGBoost Priority,0.265146,0.262895,0.508585
2,LightGBM Priority,0.260425,0.260015,0.503636


In [19]:
# Save Models
Path('../models/priority/').mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    lgb_priority,
    '../models/priority/lgb_priority.pkl'
)

joblib.dump(
    xgb_priority,
    '../models/priority/xgb_priority.pkl'
)

joblib.dump(
    rf_priority,
    '../models/priority/rf_priority.pkl'
)

['../models/priority/rf_priority.pkl']

In [20]:
# Save Results
priority_model_results.to_csv(
    "../model_results/priority_classical_ml_results.csv",
    index=False
)

ML Flow

In [8]:
dagshub.init(
    repo_owner='armaaz.au.stats',
    repo_name='AI-Powered-Customer-Support-Intelligence-Platform',
    mlflow=True
)

Accessing as armaaz.au.stats

Initialized MLflow to track repo "armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform"

Repository armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform initialized!

In [9]:
# Set Experiment

mlflow.set_experiment(
    "Ticket_Priority_Classification"
)

<Experiment: artifact_location='mlflow-artifacts:/23bcb16cc8c849c2b043194ba58b2828', creation_time=1781408134957, experiment_id='5', last_update_time=1781408134957, lifecycle_stage='active', name='Ticket_Priority_Classification', tags={}, workspace='default'>

In [21]:
# Log rf model
with mlflow.start_run(
    run_name="RF_Priority"
):

    mlflow.log_params(
        rf_priority.get_params()
    )


    mlflow.log_metrics(
        {
            "accuracy": rf_priority_results["Accuracy"],
            "f1_macro": rf_priority_results["F1 Macro"],
            "roc_auc": rf_priority_results["ROC-AUC"]
        }
    )


    mlflow.sklearn.log_model(
        rf_priority,
        "model"
    )

2026/06/14 09:43:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 09:44:01 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RF_Priority at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/5/runs/5d038199d32549ea9bb7985fdf6d22de
🧪 View experiment at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/5


In [22]:
# Log xgboost model
with mlflow.start_run(
    run_name="XGB_Priority"
):

    mlflow.log_params(
        xgb_priority.get_params()
    )


    mlflow.log_metrics(
        {
            "accuracy": xgb_priority_results["Accuracy"],
            "f1_macro": xgb_priority_results["F1 Macro"],
            "roc_auc": xgb_priority_results["ROC-AUC"]
        }
    )


    mlflow.sklearn.log_model(
        xgb_priority,
        "model"
    )

2026/06/14 09:45:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 09:45:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run XGB_Priority at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/5/runs/c1b4d1a3f0a74898b901195a77628b23
🧪 View experiment at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/5


In [23]:
# Log LightGBM model
with mlflow.start_run(
    run_name="RF_Priority"
):

    mlflow.log_params(
        lgb_priority.get_params()
    )


    mlflow.log_metrics(
        {
            "accuracy": lgb_priority_results["Accuracy"],
            "f1_macro": lgb_priority_results["F1 Macro"],
            "roc_auc": lgb_priority_results["ROC-AUC"]
        }
    )


    mlflow.sklearn.log_model(
        lgb_priority,
        "model"
    )

2026/06/14 09:45:39 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/14 09:45:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RF_Priority at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/5/runs/ae60d7b7c4724289a4f1a19f40d1f762
🧪 View experiment at: https://dagshub.com/armaaz.au.stats/AI-Powered-Customer-Support-Intelligence-Platform.mlflow/#/experiments/5


# Conclusion - Ticket Priority Prediction (Advanced Classical ML)

Advanced classical machine learning models were trained to predict ticket priority levels using structured customer and ticket-related features.

The implemented models included:

- Random Forest Classifier
- XGBoost Classifier
- LightGBM Classifier

The models were evaluated using Accuracy, Macro F1-Score, and ROC-AUC.

### Model Performance Summary:

| Model | Accuracy | F1 Macro | ROC-AUC |
|-------|----------|----------|---------|
| Random Forest | 25.73% | 25.69% | 49.34% |
| XGBoost | 26.51% | 26.29% | 50.86% |
| LightGBM | 26.04% | 26.00% | 50.36% |

### Key Observations:

- XGBoost achieved the best performance among the evaluated models with the highest Accuracy (26.51%), Macro F1-Score (26.29%), and ROC-AUC (50.86%).
- All advanced models showed only marginal improvement over the Decision Tree baseline, indicating that the available structured features provide limited information for accurately determining ticket priority.
- The similar precision, recall, and F1-score values across all priority classes suggest that the models are not strongly biased toward any specific priority category.
- ROC-AUC values close to 0.50 indicate that the models have limited ability to distinguish between different priority levels.

The results establish a benchmark for the Ticket Priority prediction task. Future improvements can be explored through feature engineering, hyperparameter optimization using Optuna, class-specific analysis, and explainability techniques such as SHAP to identify important factors influencing priority prediction.

All trained models, evaluation metrics, and experiment runs have been saved and logged using MLflow for reproducibility and future comparison.